In [ ]:
!pip install tiktoken

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('dataset_final.csv')

In [ ]:
df.info()

In [ ]:
# ============================================================
# CONTAGEM DE TOKENS POR CLASSE
# - Polido_IA: usa Abstract
# - Gerado_IA: usa Title + Introduction + Conclusion
# - Não inclui tokens do prompt/comando
# ============================================================

In [ ]:
# Escolha uma codificação compatível com modelos recentes
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))

# Cópia de segurança
df_tokens = df.copy()

# ------------------------------------------------------------
# POLIDO_IA — conta tokens apenas do Abstract
# ------------------------------------------------------------

mask_polido = df_tokens["Class"].eq("Polida_IA")

df_tokens.loc[mask_polido, "input_text_for_tokens"] = (
    df_tokens.loc[mask_polido, "Abstract"].fillna("")
)

# ------------------------------------------------------------
# GERADO_IA — conta tokens de Title + Introduction + Conclusion
# ------------------------------------------------------------

mask_gerado = df_tokens["Class"].eq("Gerada")

df_tokens.loc[mask_gerado, "input_text_for_tokens"] = (
    "Title: " + df_tokens.loc[mask_gerado, "Title"].fillna("").astype(str) + "\n\n" +
    "Introduction: " + df_tokens.loc[mask_gerado, "Introduction_clean"].fillna("").astype(str) + "\n\n" +
    "Conclusion: " + df_tokens.loc[mask_gerado, "Conclusion_clean"].fillna("").astype(str)
)

# ------------------------------------------------------------
# CONTAGEM DE TOKENS
# ------------------------------------------------------------

df_tokens["input_tokens"] = df_tokens["input_text_for_tokens"].apply(count_tokens)

# ------------------------------------------------------------
# RESUMO GERAL
# ------------------------------------------------------------

summary_tokens = (
    df_tokens[df_tokens["Class"].isin(["Polida_IA", "Gerada"])]
    .groupby("Class")
    .agg(
        registros=("input_tokens", "count"),
        total_tokens=("input_tokens", "sum"),
        media_tokens=("input_tokens", "mean"),
        mediana_tokens=("input_tokens", "median"),
        min_tokens=("input_tokens", "min"),
        max_tokens=("input_tokens", "max")
    )
    .reset_index()
)

summary_tokens

### Requisição via API

In [3]:
pip install openai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   -------------------------------------- - 1.3/1.3 MB 7.8 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 7.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 26.6 MB/s  0:00:00

   ---------------- -----------------------  5/12 [distro]
   -------------------------- -------------  8/12 [pydantic]
   -------------------------- -------------  8/12 [pydantic]
   ------------------------------ ---------  9/12 [httpcore]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --- 11/12 [openai]
   ------------------------------------ --


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from openai import OpenAI

In [6]:
import openai
print(openai.__version__)

2.38.0


In [ ]:
# ============================================================
# TESTE UNITÁRIO EM BATCH — 8 AMOSTRAS
# 4 Gerada + 4 Polida_IA
# Modelo: GPT-5.4
# ============================================================

import json
import pandas as pd

client = OpenAI(api_key="")

MODEL = "gpt-5.4"

# ------------------------------------------------------------
# 1. Amostragem
# ------------------------------------------------------------

df_gerada_test = (
    df[df["Class"].eq("Gerada")]
    .sample(n=4, random_state=42)
    .copy()
)

df_polida_test = (
    df[df["Class"].eq("Polida_IA")]
    .sample(n=4, random_state=42)
    .copy()
)

# ------------------------------------------------------------
# 2. Prompts
# ------------------------------------------------------------

def build_prompt_gerada(row, min_mean_words=101, max_mean_words=114):
    return f"""
Escreva um resumo científico em português do Brasil com base apenas nas informações fornecidas.

Não adicione informações externas.
Não invente métodos, resultados ou conclusões.
Use linguagem acadêmica formal.
O resumo deve ter entre {min_mean_words} e {max_mean_words} palavras.
Retorne apenas o resumo.

Título:
{row["Title"]}

Introdução:
{row["Introduction"]}

Conclusão:
{row["Conclusion"]}
""".strip()


POLISH_PROMPTS = [
    "Melhore a escrita deste resumo acadêmico:",
    "Reescreva este resumo de forma mais clara e acadêmica:",
    "Corrija e melhore a redação deste texto acadêmico:",
    "Deixe este resumo mais fluido e formal:"
]


def build_prompt_polida(text, prompt_instruction, min_mean_words=101, max_mean_words=114):
    return f"""
{prompt_instruction}

{text}

O resumo deve ter entre {min_mean_words} e {max_mean_words} palavras.
Retorne apenas o resumo.

""".strip()


# ------------------------------------------------------------
# 3. Criar metadados locais + requisições Batch
# ------------------------------------------------------------

requests = []
metadata_rows = []

# Classe Gerada
for _, row in df_gerada_test.iterrows():
    custom_id = f"gerada_{row['index']}"
    prompt = build_prompt_gerada(row)

    requests.append({
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "input": prompt,
            "temperature": 0.4,
            "max_output_tokens": 500
        }
    })

    metadata_rows.append({
        "custom_id": custom_id,
        "index": row["index"],
        "Class": row["Class"],
        "prompt_id": "gerada_fixo",
        "prompt_instruction": None,
        "prompt_text": prompt,
        "input_title": row["Title"],
        "input_abstract": row["Abstract"],
        "input_introduction": row["Introduction"],
        "input_conclusion": row["Conclusion"]
    })


# Classe Polida_IA
for i, (_, row) in enumerate(df_polida_test.iterrows()):
    prompt_instruction = POLISH_PROMPTS[i % len(POLISH_PROMPTS)]
    custom_id = f"polida_{i+1}_{row['index']}"
    prompt = build_prompt_polida(row["Abstract"], prompt_instruction)

    requests.append({
        "custom_id": custom_id,
        "method": "POST",
        "url": "/v1/responses",
        "body": {
            "model": MODEL,
            "input": prompt,
            "temperature": 0.4,
            "max_output_tokens": 500
        }
    })

    metadata_rows.append({
        "custom_id": custom_id,
        "index": row["index"],
        "Class": row["Class"],
        "prompt_id": f"polida_prompt_{i+1}",
        "prompt_instruction": prompt_instruction,
        "prompt_text": prompt,
        "input_title": row["Title"],
        "input_abstract": row["Abstract"],
        "input_introduction": row["Introduction"],
        "input_conclusion": row["Conclusion"]
    })


df_batch_metadata = pd.DataFrame(metadata_rows)

df_batch_metadata.to_csv(
    "metadata_teste_batch_8_amostras.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Salvar arquivo JSONL para Batch
# ------------------------------------------------------------

batch_file_path = "batch_teste_8_amostras.jsonl"

with open(batch_file_path, "w", encoding="utf-8") as f:
    for req in requests:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"Arquivo Batch criado: {batch_file_path}")
print(f"Total de requisições: {len(requests)}")

Arquivo Batch criado: batch_teste_8_amostras.jsonl
Total de requisições: 8


In [12]:
# ------------------------------------------------------------
# 5. Upload do arquivo Batch
# ------------------------------------------------------------

batch_input_file = client.files.create(
    file=open("batch_teste_8_amostras.jsonl", "rb"),
    purpose="batch"
)

print("File ID:", batch_input_file.id)

# ------------------------------------------------------------
# 6. Criar o Batch
# ------------------------------------------------------------

batch = client.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/responses",
    completion_window="24h",
    metadata={
        "description": "teste_unitario_8_amostras_gpt54"
    }
)

print("Batch ID:", batch.id)
print("Status:", batch.status)

File ID: file-SP5E7WKnHnD3vkAVmTLdPM
Batch ID: batch_6a14ab5707e48190869d369315f802f8
Status: validating


In [14]:
batch_status = client.batches.retrieve(batch.id)

print("Batch ID:", batch_status.id)
print("Status:", batch_status.status)
print("Output file ID:", batch_status.output_file_id)
print("Error file ID:", batch_status.error_file_id)
print("Request counts:", batch_status.request_counts)

Batch ID: batch_6a14ab5707e48190869d369315f802f8
Status: completed
Output file ID: file-FAYL2aDZRHTrGuBEYjFgyH
Error file ID: None
Request counts: BatchRequestCounts(completed=8, failed=0, total=8)


In [15]:
# ------------------------------------------------------------
# 7. Baixar resultados
# ------------------------------------------------------------

batch_status = client.batches.retrieve(batch.id)

output_file_id = batch_status.output_file_id

file_response = client.files.content(output_file_id)
output_text = file_response.text

with open("resultado_teste_batch_8_amostras.jsonl", "w", encoding="utf-8") as f:
    f.write(output_text)

print("Resultados salvos em resultado_teste_batch_8_amostras.jsonl")

Resultados salvos em resultado_teste_batch_8_amostras.jsonl


In [ ]:
# ------------------------------------------------------------
# 8. Ler resultados e consolidar
# ------------------------------------------------------------

outputs = []

with open("resultado_teste_batch_8_amostras.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        custom_id = item.get("custom_id")
        error = item.get("error")

        output = None
        usage = None

        if error is None:
            body = item["response"]["body"]

            # Responses API geralmente retorna output_text,
            # mas deixo fallback para estrutura com output.
            output = body.get("output_text")

            if output is None and "output" in body:
                try:
                    output = body["output"][0]["content"][0]["text"]
                except Exception:
                    output = None

            usage = body.get("usage")

        outputs.append({
            "custom_id": custom_id,
            "output": output,
            "error": error,
            "usage": usage
        })

df_outputs = pd.DataFrame(outputs)

df_test_results = df_batch_metadata.merge(
    df_outputs,
    on="custom_id",
    how="left"
)

df_test_results["output_words"] = (
    df_test_results["output"]
    .fillna("")
    .str.split()
    .str.len()
)

df_test_results.to_csv(
    "teste_batch_gpt54_8_amostras_consolidado.csv",
    index=False,
    encoding="utf-8-sig"
)

df_test_results[[
    "index",
    "Class",
    "prompt_id",
    "output_words",
    "error",
    "output"
]]

,index,Class,prompt_id,output_words,error,output
0,24931,Gerada,gerada_fixo,115,None,Este artigo apresenta um estudo exploratório s...
1,24975,Gerada,gerada_fixo,109,None,Este trabalho apresenta um resumo das experiên...
2,22069,Gerada,gerada_fixo,117,None,Este artigo descreve a experiência do projeto ...
3,17736,Gerada,gerada_fixo,112,None,OrbitAndo é uma plataforma digital de aprendiz...
4,24918,Polida_IA,polida_prompt_1,113,None,Claro — aqui está uma versão com escrita mais ...
5,24941,Polida_IA,polida_prompt_2,220,None,"Claro — segue uma versão mais clara, coesa e c..."
6,22018,Polida_IA,polida_prompt_3,126,None,Claro — segue uma versão corrigida e com redaç...
7,17395,Polida_IA,polida_prompt_4,258,None,Claro — segue uma versão mais fluida e formal ...
